In [9]:
import dagshub
import mlflow
from mlflow.tracking import MlflowClient


dagshub.init(repo_owner='kamrankhanorakzai', repo_name='solar-performance-monitoring', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/kamrankhanorakzai/solar-performance-monitoring.mlflow")
mlflow.set_experiment('solar-performance-monitoring.v4')


def register_model(model_name: str, model_info: dict):
    try:
        model_uri = f"runs:/{model_info['run_id']}/{model_info['model_path']}"

        model_version = mlflow.register_model(model_uri, model_name)

        client = MlflowClient()

        # ✅ NEW WAY (Alias instead of Stage)
        client.set_registered_model_alias(
            name=model_name,
            alias="staging",
            version=model_version.version
        )

        return model_version

    except Exception as e:
        print(f"❌ Registration failed: {e}")
        return None

register_model("final_model", {"run_id": "675af920e16148b0aa11572c806b8a82", "model_path": "final_model"})






  
  






Initialized MLflow to track repo "kamrankhanorakzai/solar-performance-monitoring"

Repository kamrankhanorakzai/solar-performance-monitoring initialized!

Registered model 'final_model' already exists. Creating a new version of this model...


❌ Registration failed: Unable to find a logged_model with artifact_path final_model under run 675af920e16148b0aa11572c806b8a82


In [ ]:
import dagshub
import mlflow
import time
from mlflow.tracking import MlflowClient


dagshub.init(
    repo_owner='kamrankhanorakzai',
    repo_name='solar-performance-monitoring',
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/kamrankhanorakzai/solar-performance-monitoring.mlflow"
)

mlflow.set_experiment('solar-performance-monitoring.v4')


def wait_until_ready(client, model_name, version, timeout=30):
    for _ in range(timeout):
        mv = client.get_model_version(name=model_name, version=version)
        if mv.status == "READY":
            return True
        time.sleep(1)
    return False


def register_model(model_name: str, model_info: dict):
    try:
        model_uri = f"runs:/{model_info['run_id']}/{model_info['model_path']}"

        # Register model
        model_version = mlflow.register_model(model_uri, model_name)

        client = MlflowClient()

        # 🔥 WAIT until model is ready
        ready = wait_until_ready(client, model_name, model_version.version)
        if not ready:
            raise Exception("Model not ready in time")

        # ✅ Set alias (NEW MLflow way)
        client.set_registered_model_alias(
            name=model_name,
            alias="staging",
            version=model_version.version
        )

        print(f"✅ Registered version {model_version.version} and set alias 'staging'")

        return model_version

    except Exception as e:
        print(f"❌ Registration failed: {e}")
        return None


# TEST
register_model(
    "final_model",
    {"run_id": "675af920e16148b0aa11572c806b8a82", "model_path": "model"}
)

Initialized MLflow to track repo "kamrankhanorakzai/solar-performance-monitoring"

Repository kamrankhanorakzai/solar-performance-monitoring initialized!

Successfully registered model 'final_model'.


❌ Registration failed: Unable to find a logged_model with artifact_path final_model under run 675af920e16148b0aa11572c806b8a82


In [15]:
import pandas as pd
import numpy as np
import joblib



def forecast(model, df, days=7):
    try:
        future = pd.DataFrame()
        df['date'] = pd.to_datetime(df['date'])
        last_date = df['date'].max()
        future['date'] = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=days)

        df_full = df.copy()
        predictions = []

        for i in range(days):

            current_date = future.loc[i, 'date']

            # Time
            month = current_date.month
            month_cos = np.cos(2 * np.pi * month / 12)

            # Lag
            lag1 = df_full['E-Today(KWH)'].iloc[-1]
            lag2 = df_full['E-Today(KWH)'].iloc[-2]

            # Rolling
            roll_mean_3 = df_full['E-Today(KWH)'].iloc[-3:].mean()
            roll_mean_7 = df_full['E-Today(KWH)'].iloc[-7:].mean()
            roll_max_7 = df_full['E-Today(KWH)'].iloc[-7:].max()
            roll_min_7 = df_full['E-Today(KWH)'].iloc[-7:].min()

            # Trend
            diff1 = lag1 - df_full['E-Today(KWH)'].iloc[-2]
            diff7 = lag1 - df_full['E-Today(KWH)'].iloc[-7]

            # Interaction
            lag1_roll7 = lag1 * roll_mean_7

            X_future = pd.DataFrame([[ 
                month_cos, lag1, lag2,
                roll_mean_3,
                roll_max_7,
                roll_min_7,
                diff1,
                diff7,
                lag1_roll7
            ]], columns=["month_cos", "lag1", "lag2",
                "roll_mean_3",
                "roll_max_7",
                "roll_min_7",
                "diff1",
                "diff7",
                "lag1_roll7"])

            pred = model.predict(X_future)[0]
            predictions.append(pred)

            # Append back
            new_row = pd.DataFrame({
                'date': [current_date],
                'E-Today(KWH)': [pred]
            })
            df_full = pd.concat([df_full, new_row], ignore_index=True)
           
        future['prediction'] = predictions

        print("✅ Forecast completed")
        
        return future

    except Exception as e:
        print(f"❌ Forecast error: {e}")
        return None

def lodad_model(model_path):
    try:
        model = joblib.load(model_path)
        print("✅ Model loaded successfully")
        return model
    except Exception as e:
        print(f"❌ Model loading error: {e}")
        return None



model=lodad_model(r"D:\solar-performance-monitoring\models\final_model\kwh_model\model_kwh.pkl")
df=pd.read_csv(r"D:\solar-performance-monitoring\data\processed\cleaned_data\final_data_forecasting.csv")
forecasted_data = forecast(model, df)


print(forecasted_data)

✅ Model loaded successfully
✅ Forecast completed
        date  prediction
0 2026-04-06   39.979553
1 2026-04-07   33.211624
2 2026-04-08   34.691307
3 2026-04-09   38.638058
4 2026-04-10   40.120506
5 2026-04-11   43.183243
6 2026-04-12   43.067513


In [18]:
import requests

url = "http://localhost:8000/kwh/forecast"

response = requests.post(url, json={"days": 7})

print(response.json())

{'forecast': [{'date': '2026-04-06T00:00:00', 'prediction': 40.0}, {'date': '2026-04-07T00:00:00', 'prediction': 33.2}, {'date': '2026-04-08T00:00:00', 'prediction': 34.68}, {'date': '2026-04-09T00:00:00', 'prediction': 38.67}, {'date': '2026-04-10T00:00:00', 'prediction': 40.13}, {'date': '2026-04-11T00:00:00', 'prediction': 43.18}, {'date': '2026-04-12T00:00:00', 'prediction': 43.06}], 'days': 7, 'message': 'KWH forecast generated successfully for 7 days'}


In [19]:
import requests

url = "http://localhost:8000/error/forecast"

response = requests.post(url, json={"days": 7})

print(response.json())

{'forecast': [{'date': '2026-04-06T00:00:00', 'prediction': 39.0}, {'date': '2026-04-07T00:00:00', 'prediction': 12.0}, {'date': '2026-04-08T00:00:00', 'prediction': 11.0}, {'date': '2026-04-09T00:00:00', 'prediction': 11.0}, {'date': '2026-04-10T00:00:00', 'prediction': 11.0}, {'date': '2026-04-11T00:00:00', 'prediction': 11.0}, {'date': '2026-04-12T00:00:00', 'prediction': 11.0}], 'days': 7, 'message': 'Error forecast generated successfully for 7 days'}


In [16]:
import requests

url = "http://localhost:8000/kwh/model-info"

response = requests.get(url)

print(response.status_code)
print(response.json())

200
{'model_type': 'MLflow PyfuncModel', 'model_name': 'final_model_kwh', 'source': 'MLflow Registry', 'alias': 'production (or latest version)', 'required_features': ['month_cos', 'lag1', 'lag2', 'roll_mean_3', 'roll_max_7', 'roll_min_7', 'diff1', 'diff7', 'lag1_roll7'], 'feature_count': 9, 'tracking_uri': 'https://dagshub.com/kamrankhanorakzai/solar-performance-monitoring.mlflow', 'repo_owner': 'kamrankhanorakzai', 'repo_name': 'solar-performance-monitoring'}


In [17]:
import requests

url = "http://localhost:8000/error/model-info"

response = requests.get(url)

print(response.status_code)
print(response.json())

200
{'model_type': 'MLflow PyfuncModel', 'model_name': 'final_model_error', 'source': 'MLflow Registry', 'alias': 'production (or latest version)', 'required_features': ['month_cos', 'lag1', 'lag2', 'roll_mean_3', 'roll_max_7', 'roll_min_7', 'diff1', 'diff7', 'lag1_roll7'], 'feature_count': 9, 'tracking_uri': 'https://dagshub.com/kamrankhanorakzai/solar-performance-monitoring.mlflow', 'repo_owner': 'kamrankhanorakzai', 'repo_name': 'solar-performance-monitoring'}


In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

def load_data(local_path):
    """Load CSV file"""
    df = pd.read_csv(local_path)
    print(f"✅ Loaded data: {df.shape}")
    return df


def get_db_engine():
    """Create PostgreSQL connection"""
    try:
        db_user = os.getenv("AWS_USER")
        db_password = os.getenv("AWS_PASS")
        db_host = os.getenv("AWS_HOST")
        db_port = os.getenv("AWS_PORT")
        db_name = os.getenv("AWS_DB")
        
        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
        )

        print("✅ Database connection successful")
        return engine

    except Exception as e:
        print(f"❌ DB connection error: {e}")
        return None


def upload_to_postgres(df, table_name, engine):
    """Upload dataframe to PostgreSQL"""
    try:
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists="replace",   # replace table each time
            index=False
        )

        print(f"🚀 Upload successful to table: {table_name}")

    except Exception as e:
        print(f"❌ Upload failed: {e}")


# -------------------------------
# MAIN


local_file = r"D:\solar-performance-monitoring\data\processed\cleaned_data\final_data_forecasting.csv"

table_name = "final_data_forecasting"

# Step 1: Load data
df = load_data(local_file)

# Step 2: Connect DB
engine = get_db_engine()

# Step 3: Upload
if engine:
    upload_to_postgres(df, table_name, engine)

✅ Loaded data: (507, 8)
✅ Database connection successful
🚀 Upload successful to table: final_data_forecasting


In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load environment variables
load_dotenv()


# -------------------------------
# DATABASE CONNECTION (AWS RDS)
# -------------------------------
def get_db_engine():
    """Create AWS RDS PostgreSQL connection"""
    try:
        db_user = os.getenv("AWS_USER")
        db_password = os.getenv("AWS_PASS")
        db_host = os.getenv("AWS_HOST")
        db_port = os.getenv("AWS_PORT", 5432)
        db_name = os.getenv("AWS_DB")

        # safety check (IMPORTANT)
        if not all([db_user, db_password, db_host, db_name]):
            raise ValueError("Missing AWS DB environment variables")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
        )

        print("✅ Connected to AWS RDS PostgreSQL")
        return engine

    except Exception as e:
        print(f"❌ DB connection error: {e}")
        return None


# -------------------------------
# LOAD DATA FROM AWS RDS
# -------------------------------
def load_from_postgres(table_name, engine):
    """Fetch data from PostgreSQL table"""
    try:
        query = f"SELECT * FROM {table_name}"
        df = pd.read_sql(query, engine)

        print(f"📥 Data loaded from DB: {df.shape}")
        return df

    except Exception as e:
        print(f"❌ Error loading data from DB: {e}")
        return None






# -------------------------------

engine = get_db_engine()

if engine:



# 2️⃣ Load data back from AWS RDS
 df_db = load_from_postgres("final_data_forecasting", engine)



✅ Connected to AWS RDS PostgreSQL
📥 Data loaded from DB: (507, 8)
         date  E-Today(KWH)  A0-Grid over voltage  A1-Grid under voltage  \
0  2024-11-16             8                   0.0                    0.0   
1  2024-11-17            40                   0.0                    0.0   
2  2024-11-18            39                   0.0                    0.0   
3  2024-11-19            39                   0.0                    1.0   
4  2024-11-20            36                   0.0                    0.0   

   A2-Grid absent  A3-Grid over frequency  A4-Grid under frequency  \
0             0.0                     0.0                      0.0   
1             0.0                     0.0                      0.0   
2             0.0                     0.0                      0.0   
3             1.0                     0.0                      1.0   
4             0.0                     0.0                      0.0   

   A6-Grid abnormal  
0               0.0  
1           